In [ ]:

import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

# 1. Load training dataset
train_data = tf.keras.utils.image_dataset_from_directory(
    'seg_train/seg_train',
    image_size=(224,224),
    batch_size=32
)

# 2. Load testing dataset
test_data = tf.keras.utils.image_dataset_from_directory(
    'seg_test/seg_test',
    image_size=(224,224),
    batch_size=32
)

num_classes = len(train_data.class_names)

# 3. Load pretrained VGG16
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# Freeze layers
base_model.trainable = False

# 4. Create model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224,224,3)),
    tf.keras.layers.Lambda(preprocess_input),
    base_model,
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

# 5. Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 6. Train model
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=5
)

# 7. Evaluate model
loss, accuracy = model.evaluate(test_data)
print("Test Accuracy:", accuracy)

# 8. Plot accuracy
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.legend(['Training', 'Validation'])
plt.show()

# 9. Plot loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.legend(['Training', 'Validation'])
plt.show()

# 10. Fine tuning
base_model.trainable = True

for layer in base_model.layers[:-4]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_data,
    validation_data=test_data,
    epochs=3
)

# 11. Evaluate after fine tuning
loss, accuracy = model.evaluate(test_data)
print("Fine Tuned Accuracy:", accuracy)
